In [27]:
%load_ext autoreload
%autoreload 2

In [1]:
import datetime
import tqdm

print(datetime.datetime.now().isoformat())

2025-06-16T23:35:04.957882


In [2]:
import bioacoustics_model_zoo as bmz

2025-06-16 23:35:20.393134: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-16 23:35:20.411405: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-16 23:35:20.432129: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-16 23:35:20.438730: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-16 23:35:20.455463: I tensorflow/core/platform/cpu_feature_guar

In [3]:
from pathlib import Path

audio = sorted(Path("~/scratch/birdclef/raw/birdclef-2024/train_audio/asbfly").expanduser().glob("*.ogg"))[
    :2
]
audio

[PosixPath('/storage/home/hcoda1/7/acheung46/scratch/birdclef/raw/birdclef-2024/train_audio/asbfly/XC134896.ogg'),
 PosixPath('/storage/home/hcoda1/7/acheung46/scratch/birdclef/raw/birdclef-2024/train_audio/asbfly/XC164848.ogg')]

In [4]:
perch = bmz.list_models()["Perch"]()
perch

/storage/scratch1/7/acheung46/birdclef/.venv/lib64/python3.9/site-packages/tensorflow_hub/__init__.py:61: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import parse_version
/storage/scratch1/7/acheung46/birdclef/.venv/lib64/python3.9/site-packages/opensoundscape/ml/cnn.py:599: UserWarning: 
                    This architecture is not listed in opensoundscape.ml.cnn_architectures.ARCH_DICT.
                    It will not be available for loading after saving the model with .save() (unless using pickle=True). 
                    To make it re-loadable, define a function that generates the architecture from arguments: (n_classes, n_channels) 
                    then use opensoundscape.ml.cnn_architectures.register_architecture() to register the generating function.

                    The function can also set the returned object's .constructor_name to the registered string key in ARCH_DICT


Perch(
  (network): MLPClassifier(
    (hidden_layers): Sequential()
    (classifier): Linear(in_features=1280, out_features=10932, bias=True)
  )
  (loss_fn): BCEWithLogitsLoss_hot()
)

In [5]:
birdnet = bmz.list_models()["BirdNET"]()
birdnet

File BirdNET_GLOBAL_6K_V2.4_Labels_af.txt already exists; skipping download.


/storage/scratch1/7/acheung46/birdclef/.venv/lib64/python3.9/site-packages/opensoundscape/ml/cnn.py:599: UserWarning: 
                    This architecture is not listed in opensoundscape.ml.cnn_architectures.ARCH_DICT.
                    It will not be available for loading after saving the model with .save() (unless using pickle=True). 
                    To make it re-loadable, define a function that generates the architecture from arguments: (n_classes, n_channels) 
                    then use opensoundscape.ml.cnn_architectures.register_architecture() to register the generating function.

                    The function can also set the returned object's .constructor_name to the registered string key in ARCH_DICT
                    to avoid this warning and ensure it is reloaded correctly by opensoundscape.ml.load_model().

                    See opensoundscape.ml.cnn_architectures module for examples of constructor functions
                    
  warnings.warn(
/storage/s

downloading model from URL...
File BirdNET_GLOBAL_6K_V2.4_Model_FP16.tflite already exists; skipping download.


BirdNET(
  (network): MLPClassifier(
    (hidden_layers): Sequential()
    (classifier): Linear(in_features=1024, out_features=6522, bias=True)
  )
  (loss_fn): BCEWithLogitsLoss_hot()
)

In [6]:
# NOTE: there is a warmup period for the model...
birdnet.predict(audio[0:1])
%time _ = birdnet.predict(audio)

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

CPU times: user 1.48 s, sys: 4.12 ms, total: 1.49 s
Wall time: 1.54 s


In [7]:
%time _ = birdnet.embed(audio, return_preds=False, clip_step=1)

  0%|          | 0/38 [00:00<?, ?it/s]

CPU times: user 3.86 s, sys: 33.5 ms, total: 3.9 s
Wall time: 3.92 s


In [8]:
# NOTE: there is a warmup period for the model...
perch.predict(audio[0:1])
%time _ = perch.predict(audio)

  0%|          | 0/5 [00:00<?, ?it/s]

I0000 00:00:1750131539.825091 2836388 service.cc:146] XLA service 0x5555746a3080 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750131539.967671 2836388 service.cc:154]   StreamExecutor device (0): Host, Default Version
2025-06-16 23:39:07.049569: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1750131553.808833 2836388 assert_op.cc:38] Ignoring Assert operator jax2tf_infer_fn_/assert_equal_1/Assert/AssertGuard/Assert
I0000 00:00:1750131577.531276 2836388 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  0%|          | 0/8 [00:00<?, ?it/s]

CPU times: user 11.3 s, sys: 23 ms, total: 11.3 s
Wall time: 10.4 s


In [73]:
from birdclef.kaggle.compile import load_tflite_interpreter, run_perch_tflite
import tensorflow as tf
from contexttimer import Timer

# Path to the TFLite model
perch_tflite_path = Path("~/scratch/birdclef/models/2025/v1/Perch/torch-linear-v1/perch.tflite").expanduser()

# Load the TFLite interpreter
perch_interpreter = load_tflite_interpreter(perch_tflite_path)

# First run to warm up
perch_dataloader = perch.predict_dataloader(audio[0:1])
_ = run_perch_tflite(perch_interpreter, perch_dataloader)

# Measure execution time
def run_perch():
    perch_dataloader = perch.predict_dataloader(audio)
    return run_perch_tflite(perch_interpreter, perch_dataloader)

%time perch_out = run_perch()

2025-06-17 01:15:34.816675: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_inter_op_parallelism which is not in the op definition: Op<name=Transpose; signature=x:T, perm:Tperm -> y:T; attr=T:type; attr=Tperm:type,default=DT_INT32,allowed=[DT_INT32, DT_INT64]> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node Transpose}}


CPU times: user 846 ms, sys: 6.69 ms, total: 852 ms
Wall time: 847 ms


In [43]:
perch_out

0     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.042605   
                                                   5.0        10.0      0.043925   
                                                   10.0       15.0      0.191889   
                                                   15.0       20.0      0.078761   
                                                   20.0       25.0     -0.008164   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.014053   
                                                   5.0        10.0      0.100804   
                                                   10.0       15.0      0.064618   

                                                                            1     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.141005   
                                                   5.0        10.0     -0.158573   
                                                   10.0       15.0     -0.165586   
                                                   15.0       20.0     -0.127910   
                                                   20.0       25.0     -0.166585   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.047585   
                                                   5.0        10.0     -0.103619   
                                                   10.0       15.0     -0.125703   

                                                                            2     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0       0.025648   
                                                   5.0        10.0      0.057496   
                                                   10.0       15.0      0.032176   
                                                   15.0       20.0      0.067172   
                                                   20.0       25.0      0.077055   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0       0.023723   
                                                   5.0        10.0      0.048283   
                                                   10.0       15.0      0.040848   

                                                                            3     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0       0.041677   
                                                   5.0        10.0      0.006564   
                                                   10.0       15.0      0.041329   
                                                   15.0       20.0     -0.001162   
                                                   20.0       25.0      0.020314   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.023687   
                                                   5.0        10.0     -0.017900   
                                                   10.0       15.0     -0.039507   

                                                                            4     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0       0.219057   
                                                   5.0        10.0      0.133760   
                                                   10.0       15.0      0.241032   
                                                   15.0       20.0      0.362048   
                                                   20.0       25.0      0.305957   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0       0.064599   
                                        

In [40]:
perch_out.shape

(8, 1280)

In [10]:
perch_interpreter.get_input_details()

[{'name': 'serving_default_inputs:0',
  'index': 0,
  'shape': array([     1, 160000], dtype=int32),
  'shape_signature': array([    -1, 160000], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}}]

In [11]:
perch_interpreter.get_output_details()

[{'name': 'StatefulPartitionedCall:2',
  'index': 384,
  'shape': array([  1, 500, 160], dtype=int32),
  'shape_signature': array([ -1, 500, 160], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}},
 {'name': 'StatefulPartitionedCall:0',
  'index': 828,
  'shape': array([   1, 1280], dtype=int32),
  'shape_signature': array([  -1, 1280], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}},
 {'name': 'StatefulPartitionedCall:3',
  'index': 832,
  'shape': array([   1, 2333], dtype=int32),
  'shape_signature': array([  -1, 2333], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'

In [58]:
from birdclef.kaggle.compile import load_tflite_interpreter
import tensorflow as tf
from contexttimer import Timer

# Path to the TFLite model
birdnet_tflite_path = Path("~/scratch/birdclef/models/2025/v1/BirdNET/torch-linear-v1/birdnet.tflite").expanduser()

# Load the TFLite interpreter
birdnet_interpreter = load_tflite_interpreter(birdnet_tflite_path)

In [13]:
birdnet_input_details = birdnet_interpreter.get_input_details()
birdnet_output_details = birdnet_interpreter.get_output_details()

In [14]:
birdnet_input_details

[{'name': 'INPUT',
  'index': 0,
  'shape': array([     1, 144000], dtype=int32),
  'shape_signature': array([    -1, 144000], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}}]

In [15]:
birdnet_output_details

[{'name': 'Identity',
  'index': 546,
  'shape': array([   1, 6522], dtype=int32),
  'shape_signature': array([  -1, 6522], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}}]

In [ ]:
birdnet_dataloader = birdnet.predict_dataloader(audio[0:1], clip_step=1)

res = []
for batch in birdnet_dataloader:
    birdnet_interpreter.set_tensor(birdnet_input_details[0]['index'], batch[0])
    birdnet_interpreter.invoke()
    output_data = birdnet_interpreter.get_tensor(birdnet_output_details[0]['index'])
    res.append(output_data)

res[0].shape


(1, 6522)

In [60]:
import numpy as np
import pandas as pd

pd.DataFrame(
    data=np.stack(res).squeeze(),
    index=birdnet_dataloader.dataset.dataset.label_df.index,
)

0     \
file                                               start_time end_time              
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        3.0       -7.506774   
                                                   1.0        4.0       -8.647613   
                                                   2.0        5.0       -7.845464   
                                                   3.0        6.0       -7.643373   
                                                   4.0        7.0       -9.401468   
                                                   5.0        8.0       -8.580341   
                                                   6.0        9.0      -10.213643   
                                                   7.0        10.0      -6.160130   
                                                   8.0        11.0     -10.190626   
                                                   9.0        12.0     -10.497392   
                                                   10.0       13.0     -11.946979   
                                                   11.0       14.0      -9.964238   
                                                   12.0       15.0      -7.970989   
                                                   13.0       16.0      -9.138251   
                                                   14.0       17.0     -10.244061   
                                                   15.0       18.0      -9.551040   
                                                   16.0       19.0      -8.209714   
                                                   17.0       20.0      -7.096784   
                                                   18.0       21.0     -11.533674   
                                                   19.0       22.0     -12.320047   
                                                   20.0       23.0     -12.408878   
                                                   21.0       24.0     -10.641862   
                                                   22.0       25.0     -10.671396   
                                                   23.0       26.0     -13.191398   
                                                   24.0       27.0     -11.255970   

                                                                             1     \
file                                               start_time end_time              
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        3.0      -12.735567   
                                                   1.0        4.0      -14.743239   
                                                   2.0        5.0      -11.027521   
                                                   3.0        6.0      -11.929975   
                                                   4.0        7.0      -13.477317   
                                                   5.0        8.0      -15.670010   
                                                   6.0        9.0      -15.701523   
                                                   7.0        10.0     -10.711525   
                                                   8.0        11.0     -13.554422   
                                                   9.0        12.0     -15.333166   
                                                   10.0       13.0     -15.949151   
                                                   11.0       14.0      -9.687764   
                                                   12.0       15.0     -10.515049   
                                                   13.0       16.0     -12.599579   
                                                   14.0       17.0     -14.244036   
                                                   15.0       18.0     -13.459959   
                                                   16.0       19.0     -10.374965   
                                                   17.0       20.0     -11.094476   
                                                   18.0       21.0     -15.977875

In [ ]:
from birdclef.kaggle.compile import run_birdnet_tflite

# birdnet_dataloader = birdnet.predict_dataloader(audio[0:1])
# _ = run_birdnet_tflite(birdnet_interpreter, birdnet_dataloader)

def run_birdnet(interpreter, clip_step=None):
    birdnet_dataloader = birdnet.predict_dataloader(audio, clip_step=clip_step)
    return run_birdnet_tflite(interpreter, birdnet_dataloader)

%time _ = run_birdnet(birdnet_interpreter)
%time _ = run_birdnet(birdnet_interpreter, clip_step=1)

CPU times: user 824 ms, sys: 16.5 ms, total: 840 ms
Wall time: 849 ms
CPU times: user 2.24 s, sys: 28.1 ms, total: 2.27 s
Wall time: 2.29 s


In [79]:
birdnet_interpreter_multi = load_tflite_interpreter(birdnet_tflite_path, num_threads=8)

%time _ = run_birdnet(birdnet_interpreter_multi)
%time _ = run_birdnet(birdnet_interpreter_multi, clip_step=1)

CPU times: user 1.66 s, sys: 39.6 ms, total: 1.7 s
Wall time: 770 ms
CPU times: user 3.83 s, sys: 31.8 ms, total: 3.87 s
Wall time: 1.86 s
